# AgriMitra Farm Analytics: Motala, Buldhana
This notebook builds an end-to-end analytics stack for a farm in Motala (Buldhana), including SQL ingestion of IoT sensor data, price and weather forecasting, crop yield prediction, geospatial visualization, and a Gemini-powered chatbot for advisory.

In [ ]:
# 1) Import Dependencies and Notebook Configuration
import os, sys, json, math, time, random, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
\n# viz
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
\n# sql + io
from sqlalchemy import create_engine, text
\n# ml
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib
\n# time series
from statsmodels.tsa.seasonal import STL
\n# geo
import folium
\n# widgets
try:
    import ipywidgets as widgets
except Exception:
    widgets = None
\n# env
from dotenv import load_dotenv
load_dotenv()
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
OPENWEATHER_API_KEY = os.getenv('OPENWEATHER_API_KEY')
DATABASE_URL = os.getenv('DATABASE_URL', 'sqlite:///farm.db')
random.seed(42); np.random.seed(42)
print('Environment set. DB=', DATABASE_URL)

In [ ]:
# 2) Farm Metadata and Geolocation for Motala, Buldhana
from dataclasses import dataclass
@dataclass
class Farm:
    id: int
    name: str
    lat: float
    lon: float
    area_ha: float
\nMOTALA_FARM = Farm(id=1, name='Ramesh Farm (Motala, Buldhana)', lat=20.53, lon=76.18, area_ha=3.2)
MOTALA_FARM

In [ ]:
# 3) Local SQL Database Setup (SQLite + SQLAlchemy)
engine = create_engine(DATABASE_URL)
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS farms (id INTEGER PRIMARY KEY, name TEXT, lat REAL, lon REAL, area_ha REAL)
    """))
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS sensor_readings (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            farm_id INTEGER, timestamp TEXT, device_id TEXT,
            metric TEXT, value REAL, unit TEXT, zone TEXT
        )
    """))
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS weather (
            ts TEXT, farm_id INTEGER, temp REAL, humidity REAL, wind REAL, rain REAL
        )
    """))
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS market_prices (
            date TEXT, commodity TEXT, market TEXT, price REAL
        )
    """))
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS yield_records (
            year INTEGER, farm_id INTEGER, crop TEXT, yield_q_per_acre REAL, fertilizer_kg REAL, irrigation_mm REAL
        )
    """))
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS model_registry (
            model_name TEXT, version TEXT, created_at TEXT, params TEXT, metrics TEXT
        )
    """))
\n# upsert farm
with engine.begin() as conn:
    conn.execute(text("INSERT OR REPLACE INTO farms (id,name,lat,lon,area_ha) VALUES (:id,:name,:lat,:lon,:area_ha)"),
                 dict(id=MOTALA_FARM.id, name=MOTALA_FARM.name, lat=MOTALA_FARM.lat, lon=MOTALA_FARM.lon, area_ha=MOTALA_FARM.area_ha))
print('DB initialized')

In [ ]:
# 4) Sensor Data Ingestion Function to SQL (IMPORTANT)
from typing import List
SENSOR_SCHEMA = ['timestamp','device_id','metric','value','unit','zone']
def ingest_sensor_batch(farm_id: int, payload_df: pd.DataFrame) -> int:
    df = payload_df.copy()
    missing = [c for c in SENSOR_SCHEMA if c not in df.columns]
    if missing:
        raise ValueError(f'Missing columns: {missing}')
    # coerce types
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df = df.dropna(subset=['timestamp','value'])
    # basic range checks (example)
    df = df[(df['value'] > -1e6) & (df['value'] < 1e6)]
    df['farm_id'] = farm_id
    # drop duplicates
    df = df.drop_duplicates(['farm_id','timestamp','device_id','metric','zone'])
    with engine.begin() as conn:
        df.assign(timestamp=df['timestamp'].astype(str)).to_sql('sensor_readings', conn, if_exists='append', index=False)
    return len(df)
\n# demo ingest
demo = pd.DataFrame({
    'timestamp': pd.date_range('2025-10-10', periods=24, freq='H'),
    'device_id': 'soil-01',
    'metric': 'soil_moisture',
    'value': np.random.uniform(35,55,24),
    'unit': '%',
    'zone': 'A'
})
rows = ingest_sensor_batch(MOTALA_FARM.id, demo)
rows

In [ ]:
# 5) External Weather API Client and Cache
import pathlib, hashlib
CACHE_DIR = pathlib.Path('artifacts/cache'); CACHE_DIR.mkdir(parents=True, exist_ok=True)
def _cache_key(lat, lon, start, end):
    h = hashlib.md5(f'{lat}-{lon}-{start}-{end}'.encode()).hexdigest()
    return CACHE_DIR / f'weather_{h}.parquet'
\nimport requests
def fetch_weather(lat: float, lon: float, start: str, end: str, api_key: str = OPENWEATHER_API_KEY) -> pd.DataFrame:
    cache = _cache_key(lat, lon, start, end)
    if cache.exists():
        return pd.read_parquet(cache)
    if not api_key:
        # synthesize data for demo
        rng = pd.date_range(start, end, freq='H')
        df = pd.DataFrame({'ts': rng, 'temp': 28 + 4*np.sin(np.arange(len(rng))/24*2*np.pi), 'humidity': 55+10*np.random.randn(len(rng)), 'wind': np.abs(np.random.randn(len(rng))*2), 'rain': np.clip(np.random.randn(len(rng)),0,None)})
        df.to_parquet(cache, index=False)
        return df
    # Example: use One Call History (requires paid plan) - here we simulate hourly pulls
    rng = pd.date_range(start, end, freq='H')
    df = pd.DataFrame({'ts': rng, 'temp': 30 + np.random.randn(len(rng)), 'humidity': 60 + 5*np.random.randn(len(rng)), 'wind': np.abs(np.random.randn(len(rng))*2), 'rain': np.clip(np.random.randn(len(rng)),0,None)})
    df.to_parquet(cache, index=False)
    return df
\nweather_df = fetch_weather(MOTALA_FARM.lat, MOTALA_FARM.lon, '2025-09-15', '2025-10-14')
weather_df.head()

In [ ]:
# 6) ETL and Feature Engineering Pipeline (IMPORTANT)

def build_features(farm_id: int, start: str, end: str) -> pd.DataFrame:
    with engine.begin() as conn:
        sens = pd.read_sql(text("SELECT * FROM sensor_readings WHERE farm_id=:fid"), conn, params={'fid': farm_id})
    sens['timestamp'] = pd.to_datetime(sens['timestamp'])
    sens = sens[(sens['timestamp']>=start)&(sens['timestamp']<=end)]
    pivot = sens.pivot_table(index='timestamp', columns='metric', values='value', aggfunc='mean').sort_index()
    w = weather_df.set_index('ts').sort_index()
    df = pivot.join(w, how='outer').interpolate().ffill().bfill()
    # basic time features
    df['hour'] = df.index.hour; df['dow'] = df.index.dayofweek; df['month'] = df.index.month
    # lags/rolls
    for c in ['soil_moisture','temp','humidity','rain']:
        if c in df.columns:
            df[f'{c}_lag1'] = df[c].shift(1)
            df[f'{c}_roll6'] = df[c].rolling(6).mean()
    df = df.dropna()
    return df

features = build_features(MOTALA_FARM.id, '2025-09-20','2025-10-14')
features.head()

In [ ]:
# 7) Geospatial Map of Farm Location (Folium)
m = folium.Map(location=[MOTALA_FARM.lat, MOTALA_FARM.lon], zoom_start=12)
folium.Marker([MOTALA_FARM.lat, MOTALA_FARM.lon], tooltip=MOTALA_FARM.name).add_to(m)
m

In [ ]:
# 8) Exploratory Visualizations: Trends and Anomalies
def rolling_zscore(s, win=24):
    m = s.rolling(win).mean(); sd = s.rolling(win).std()
    return (s - m) / (sd + 1e-6)
\nfig = go.Figure()
if 'soil_moisture' in features.columns:
    z = rolling_zscore(features['soil_moisture'])
    fig.add_trace(go.Scatter(x=features.index, y=features['soil_moisture'], name='Soil Moisture'))
    fig.add_trace(go.Scatter(x=features.index, y=z, name='Z-Score', yaxis='y2'))
fig.update_layout(title='Sensor Trends (with anomaly score)', yaxis2=dict(overlaying='y', side='right'))
fig

In [ ]:
# 9) Market Price Data Load and Cleaning
csv_path = '../data/mandi/sample_prices.csv'
prices = pd.read_csv(csv_path, parse_dates=['date'])
prices = prices.sort_values('date')
# winsorize via IQR
q1,q3 = prices['price'].quantile([0.25,0.75]); iqr = q3-q1
low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
prices['price'] = prices['price'].clip(low, high)
prices.head()

In [ ]:
# 10) Price Trend Analysis (Rolling, STL)
prices_idx = prices.set_index('date')
stl = STL(prices_idx['price'], period=7, robust=True).fit()
fig,axs = plt.subplots(3,1, figsize=(10,6), sharex=True)
axs[0].plot(prices_idx.index, stl.trend); axs[0].set_title('Trend')
axs[1].plot(prices_idx.index, stl.seasonal); axs[1].set_title('Seasonality')
axs[2].plot(prices_idx.index, stl.resid); axs[2].set_title('Residual')
plt.tight_layout(); plt.show()

In [ ]:
# 11) Price Prediction Model (IMPORTANT)
def make_price_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy(); df['date'] = pd.to_datetime(df['date']); df = df.sort_values('date')
    df['lag1'] = df['price'].shift(1); df['lag7'] = df['price'].shift(7)
    df['roll3'] = df['price'].rolling(3).mean(); df['roll7'] = df['price'].rolling(7).mean()
    df['dow'] = df['date'].dt.dayofweek; df['month'] = df['date'].dt.month
    return df.dropna()
\nfeat = make_price_features(prices)
X = feat[['lag1','lag7','roll3','roll7','dow','month']].values
y = feat['price'].values
tscv = TimeSeriesSplit(n_splits=3)
rmses, mapes = [], []
model = RandomForestRegressor(n_estimators=300, random_state=42)
for train_idx, test_idx in tscv.split(X):
    model.fit(X[train_idx], y[train_idx])
    pred = model.predict(X[test_idx])
    rmses.append(mean_squared_error(y[test_idx], pred, squared=False))
    mapes.append(mean_absolute_percentage_error(y[test_idx], pred))
print('CV RMSE:', np.mean(rmses), 'CV MAPE:', np.mean(mapes))
\n# Train on full and forecast 7 days iteratively
model.fit(X, y)
def forecast_price(model, df: pd.DataFrame, horizon=7):
    work = df.copy()
    preds = []
    for _ in range(horizon):
        f = make_price_features(work).tail(1)
        X_last = f[['lag1','lag7','roll3','roll7','dow','month']].values
        p = float(model.predict(X_last)[0])
        last_date = pd.to_datetime(work['date']).max()
        next_date = last_date + pd.Timedelta(days=1)
        work = pd.concat([work, pd.DataFrame({'date':[next_date], 'price':[p]})], ignore_index=True)
        preds.append(p)
    return preds, work
preds, work = forecast_price(model, prices[['date','price']], horizon=7)
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=prices['date'], y=prices['price'], name='Actual'))
future_dates = pd.date_range(prices['date'].max()+pd.Timedelta(days=1), periods=7)
fig2.add_trace(go.Scatter(x=future_dates, y=preds, name='Forecast'))
fig2.update_layout(title='Mandi Price Forecast (7-day)')
fig2

In [ ]:
# 12) Weather Forecasting Model (IMPORTANT)
def train_weather_model(ts: pd.Series):
    # simple baseline: GradientBoosting on lag features
    df = pd.DataFrame({'y': ts}).dropna()
    df['lag1'] = df['y'].shift(1)
    df['lag24'] = df['y'].shift(24)
    df['roll24'] = df['y'].rolling(24).mean()
    df = df.dropna()
    X = df[['lag1','lag24','roll24']].values
    y = df['y'].values
    m = GradientBoostingRegressor(random_state=42)
    m.fit(X, y)
    return m, df
\nwt_model, wt_df = train_weather_model(weather_df.set_index('ts')['temp'])
def forecast_weather(model, df: pd.DataFrame, horizon=48):
    work = df.copy()
    preds = []
    for _ in range(horizon):
        f = pd.DataFrame({'y': work['temp']})
        f['lag1'] = f['y'].shift(1)
        f['lag24'] = f['y'].shift(24)
        f['roll24'] = f['y'].rolling(24).mean()
        f = f.dropna()
        x_last = f[['lag1','lag24','roll24']].values[-1:]
        p = float(model.predict(x_last)[0])
        next_ts = work['ts'].max() + pd.Timedelta(hours=1)
        work = pd.concat([work, pd.DataFrame({'ts':[next_ts],'temp':[p]})], ignore_index=True)
        preds.append(p)
    return preds, work
wpreds, wwork = forecast_weather(wt_model, weather_df[['ts','temp']].copy(), horizon=48)
go.Figure(data=[go.Scatter(x=weather_df['ts'], y=weather_df['temp'], name='Actual'), go.Scatter(x=pd.date_range(weather_df['ts'].max()+pd.Timedelta(hours=1), periods=48, freq='H'), y=wpreds, name='Forecast')])

In [ ]:
# 13) Crop Yield Prediction and Optimization Scenarios (IMPORTANT)
def synthesize_yield_data(n=100):
    rng = pd.date_range('2023-01-01', periods=n, freq='7D')
    df = pd.DataFrame({
        'date': rng,
        'yield_q_per_acre': 8 + 0.02*np.arange(n) + np.random.randn(n)*0.2,
        'fertilizer_kg': np.random.uniform(10,30,n),
        'irrigation_mm': np.random.uniform(15,60,n),
        'temp_avg': 28 + np.random.randn(n),
        'rain_mm': np.abs(np.random.randn(n))*5
    })
    return df
\nyield_df = synthesize_yield_data(120)
Xy = yield_df[['fertilizer_kg','irrigation_mm','temp_avg','rain_mm']]; y_yield = yield_df['yield_q_per_acre']
y_model = GradientBoostingRegressor(random_state=42).fit(Xy, y_yield)
\n# Scenario simulator
def recommend_practices(model, base_features: dict, scenarios: list[dict], top_k=3):
    rows = []
    for sc in scenarios:
        feats = base_features.copy(); feats.update(sc)
        X = pd.DataFrame([feats])[['fertilizer_kg','irrigation_mm','temp_avg','rain_mm']]
        pred = float(model.predict(X)[0])
        rows.append({'scenario': sc, 'predicted_yield': pred})
    out = pd.DataFrame(rows).sort_values('predicted_yield', ascending=False).head(top_k)
    return out
\nbase = {'fertilizer_kg':20,'irrigation_mm':30,'temp_avg':29,'rain_mm':3}
scs = [
    {'fertilizer_kg':22,'irrigation_mm':35},
    {'fertilizer_kg':18,'irrigation_mm':28},
    {'fertilizer_kg':24,'irrigation_mm':40},
    {'fertilizer_kg':26,'irrigation_mm':32},
]
recommend_practices(y_model, base, scs, top_k=3)

In [ ]:
# 14) Interactive UI with ipywidgets
if widgets:
    farm_selector = widgets.Dropdown(options=[(MOTALA_FARM.name,MOTALA_FARM.id)], description='Farm:')
    horizon_slider = widgets.IntSlider(value=7, min=3, max=21, step=1, description='Horizon')
    display(farm_selector, horizon_slider)
else:
    print('ipywidgets not available in this environment')

In [ ]:
# 15) Chatbot Integration using Gemini API (IMPORTANT)
def chat_with_gemini(prompt: str, context: dict | None = None) -> str:
    if not GEMINI_API_KEY:
        return "Gemini API key not configured. Set GEMINI_API_KEY to enable."
    try:
        import google.generativeai as genai
        genai.configure(api_key=GEMINI_API_KEY)
        model = genai.GenerativeModel('gemini-1.5-flash')
        system = "You are AgriMitra assistant. Use provided context (if any) to answer succinctly for a farmer in Motala, Buldhana."
        ctx_json = json.dumps(context or {})[:8000]
        resp = model.generate_content(system + "\nCONTEXT:\n" + ctx_json + "\nQUESTION:\n" + prompt)
        return resp.text or "(no text)"
    except Exception as e:
        return f"Gemini error: {e}"
\n# demo call (safe even if no API key)
chat_with_gemini("How much should I water today?", {"soil_moisture":45, "temp":32})

In [ ]:
# 16) Real-time Update Loop (Simulation)
def simulate_stream(n=5, delay=0.1):
    ts0 = pd.Timestamp.utcnow().floor('H')
    rows = []
    for i in range(n):
        rows.append({'timestamp': ts0 + pd.Timedelta(hours=i), 'device_id':'soil-01','metric':'soil_moisture','value':float(40+np.random.randn()),'unit':'%','zone':'A'})
    df = pd.DataFrame(rows)
    return ingest_sensor_batch(MOTALA_FARM.id, df)
\nsimulate_stream(6)

In [ ]:
# 17) Model Persistence and Inference Utilities
ART_DIR = 'artifacts/models'; os.makedirs(ART_DIR, exist_ok=True)
joblib.dump(model, f'{ART_DIR}/mandi_rf.joblib')
joblib.dump(wt_model, f'{ART_DIR}/weather_gbr.joblib')
joblib.dump(y_model, f'{ART_DIR}/yield_gbr.joblib')
\n# predict helpers
def predict_price_next(prices_df: pd.DataFrame, horizon=7):
    preds,_ = forecast_price(model, prices_df[['date','price']], horizon=horizon)
    return preds
def predict_weather_next(hours=48):
    preds,_ = forecast_weather(wt_model, weather_df[['ts','temp']].copy(), horizon=hours)
    return preds
def predict_yield_scenarios(base, scs):
    return recommend_practices(y_model, base, scs)

In [ ]:
# 18) Minimal Tests for Core Functions
def _test_ingest():
    df = pd.DataFrame({'timestamp':[pd.Timestamp('2025-10-14')],'device_id':['t1'],'metric':['temp'],'value':[30.2],'unit':['C'],'zone':['Z']})
    n = ingest_sensor_batch(MOTALA_FARM.id, df)
    assert n == 1
    return 'ingest ok'
\n_test_ingest()

In [ ]:
# 19) Scheduler for periodic jobs (APScheduler)
from apscheduler.schedulers.background import BackgroundScheduler
sched = BackgroundScheduler()
def job_fetch_weather():
    fetch_weather(MOTALA_FARM.lat, MOTALA_FARM.lon, '2025-10-01','2025-10-14')
def job_retrain_price():
    global model
    model.fit(X, y)
sched.add_job(job_fetch_weather, 'interval', hours=6)
sched.add_job(job_retrain_price, 'interval', weeks=1)
# sched.start()  # Uncomment to run in notebook session

In [ ]:
# 20) Export Reports and Snapshots
ART_DIR = 'artifacts/exports'; os.makedirs(ART_DIR, exist_ok=True)
prices.to_csv(f'{ART_DIR}/prices_clean.csv', index=False)
features.reset_index().to_parquet(f'{ART_DIR}/features.parquet', index=False)
fig2.write_html(f'{ART_DIR}/price_forecast.html')
m.save(f'{ART_DIR}/farm_map.html')
print('Exports written to', ART_DIR)